# Explore Green Taxi data

Choose a starting month and duration, then explore those staged files. Downloads are optional; the analysis does not run the revenue pipeline or change raw records.

Use the project's `.venv` kernel after `uv sync --locked`, or the running Jupyter container's Python kernel. That container mounts this checkout at `/workspace` with writable access, so downloads appear in the host `data/raw/` folder. This is separate from the pipeline's Docker named volume.

See the [design](../docs/adr/001-pipeline-design.md) for approved data rules and the [data review](../docs/data-review.md) for the original January–March findings. Another period can have different results.

In [12]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

root = Path.cwd()
if not (root / 'pyproject.toml').is_file():
    root = root.parent
if not (root / 'pyproject.toml').is_file():
    raise FileNotFoundError('Open this notebook from the repository or notebooks folder.')
sys.path.insert(0, str(root / 'src'))

## Choose the period

Change `start_month` and `duration_months` here. The duration is a positive integer counting whole months. This selection controls downloading, loading and the date checks below; it does not edit the shared defaults.

After changing settings, rerun this cell and every following code cell. Run the download cell only when you need to stage files.

In [13]:
from green_taxi.settings import PipelineSettings

settings = PipelineSettings(
    data_dir=root / 'data',
    reporting_period={
        'start_month': '2025-03',
        'duration_months': 4,  # Number of whole months to download.
    },
)
period = settings.reporting_period
print(f'Months: {", ".join(settings.months)}')
print(f'Pickup dates: {period.start} to {period.end} (end excluded)')

Months: 2025-03, 2025-04, 2025-05, 2025-06
Pickup dates: 2025-03-01 to 2025-07-01 (end excluded)


## Download the selected months (optional)

Run this cell if files are missing. It uses the existing downloader and reuses valid staged files without refreshing them. If all files already exist, skip to loading. No data is downloaded by the load or analysis cells.

In [14]:
from green_taxi.download import TaxiDownloader

staged = TaxiDownloader(settings).download()
for item in staged:
    print(f'File ready: {item.path}')

File ready: /workspace/data/raw/green_tripdata_2025-03.parquet
File ready: /workspace/data/raw/green_tripdata_2025-04.parquet
File ready: /workspace/data/raw/green_tripdata_2025-05.parquet
File ready: /workspace/data/raw/green_tripdata_2025-06.parquet


## Load the selected files

Read every file selected by the settings above. The table shows the source filenames and row counts so you can check what is being explored. Loading does not filter out-of-period records.

In [15]:
sources = [
    settings.raw_dir / f'green_tripdata_{month}.parquet'
    for month in settings.months
]
missing = [source.name for source in sources if not source.is_file()]
if missing:
    raise FileNotFoundError(
        f'Missing taxi files: {", ".join(missing)}\n'
        'Run the Download the selected months cell, then rerun this cell.'
    )

frames = [pd.read_parquet(source) for source in sources]
rides = pd.concat(frames, ignore_index=True)
file_counts = pd.DataFrame({
    'file': [source.name for source in sources],
    'rows': [len(frame) for frame in frames],
})
print(file_counts.to_string(index=False))
print(f'Total rows loaded: {len(rides):,}')
rides.head()

                          file  rows
green_tripdata_2025-03.parquet 51539
green_tripdata_2025-04.parquet 52132
green_tripdata_2025-05.parquet 55399
green_tripdata_2025-06.parquet 49390
Total rows loaded: 208,460


,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,...,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee
0,2,2025-03-01 00:07:34,2025-03-01 00:24:52,N,1.0,75,239,1.0,2.20,18.40,...,0.5,5.91,0.00,NaN,1.0,29.56,1.0,1.0,2.75,0.0
1,2,2025-03-01 00:01:24,2025-03-01 00:10:03,N,1.0,41,42,1.0,1.06,8.60,...,0.5,3.33,0.00,NaN,1.0,14.43,1.0,1.0,0.00,0.0
2,2,2025-03-01 00:45:03,2025-03-01 01:05:38,N,1.0,265,56,1.0,18.91,69.50,...,0.5,0.00,0.00,NaN,1.0,72.00,2.0,1.0,0.00,0.0
3,2,2025-03-01 00:10:10,2025-03-01 00:29:35,N,5.0,82,236,1.0,8.36,30.17,...,0.5,3.35,6.94,NaN,1.0,44.71,1.0,1.0,2.75,0.0
4,2,2025-03-01 00:16:14,2025-03-01 00:19:44,N,1.0,66,33,1.0,0.82,5.80,...,0.5,0.00,0.00,NaN,1.0,8.30,2.0,1.0,0.00,0.0


## Which columns do we need?

The pipeline groups by pickup date and sums `total_amount`. Check the types and missing values across the selected files.

In [16]:
columns = ['lpep_pickup_datetime', 'total_amount']
pd.DataFrame({
    'type': rides[columns].dtypes.astype(str),
    'missing': rides[columns].isna().sum(),
})

,type,missing
lpep_pickup_datetime,datetime64[us],0
total_amount,float64,0


## Dates and amounts

Compare recorded pickup dates with the selected period. Negative and zero amounts are kept by the pipeline; they are not automatically errors. These are exploratory counts over all loaded rows, not the pipeline's non-overlapping exclusion counts.

In [17]:
pickup = pd.to_datetime(rides['lpep_pickup_datetime'], errors='coerce')
amount = pd.to_numeric(rides['total_amount'], errors='coerce')
period = settings.reporting_period
outside = pickup.notna() & (
    (pickup < pd.Timestamp(period.start)) | (pickup >= pd.Timestamp(period.end))
)
pd.Series({
    'selected start (included)': period.start,
    'selected end (excluded)': period.end,
    'rows loaded': len(rides),
    'first pickup': pickup.min(),
    'last pickup': pickup.max(),
    'missing or invalid pickups': pickup.isna().sum(),
    'pickups outside selected period': outside.sum(),
    'lowest amount': amount.min(),
    'highest amount': amount.max(),
    'missing, invalid or non-finite amounts': (~np.isfinite(amount)).sum(),
    'negative amounts': (amount < 0).sum(),
    'zero amounts': (amount == 0).sum(),
}, name=f'{settings.months[0]} to {settings.months[-1]}')

selected start (included)                          2025-03-01
selected end (excluded)                            2025-07-01
rows loaded                                            208460
first pickup                              2025-02-25 18:10:10
last pickup                               2025-07-01 20:35:08
missing or invalid pickups                                  0
pickups outside selected period                            10
lowest amount                                          -473.1
highest amount                                        1095.54
missing, invalid or non-finite amounts                      0
negative amounts                                          634
zero amounts                                              422
Name: 2025-03 to 2025-06, dtype: object

## What does this tell us?

A file's month does not guarantee every pickup falls in that month. The outside-period count measures this against your current selection. Changing the duration changes both the source files and the date range.

The pipeline excludes invalid/out-of-period pickups and invalid amounts, while retaining valid zero and negative amounts. This notebook inspects the raw values; it does not apply those rules or calculate revenue. Use the [data review](../docs/data-review.md) as a comparison only when exploring the same source files and January–March period.